# EXT 외관 결함 — 라벨 없는 멀티모달 파이프라인

**왜 라벨을 버리나.** 라벨이 깨져 있다는 게 확정됐다 —
`0731 §9` 육안(캡 부식·외피 찢김이 라벨 없음) · `0731 §4.13`(conf≥0.10 검출의 **46.4%가 라벨 미대응**) ·
`0803 §11`(GT가 결함 **전체가 아니라 조각**, 면적비 p50 112배) · `0803 §12`(모델 레버 7개가 전부 0.12~0.22 수렴).
= **모델 천장이 아니라 잣대의 천장.**

**그 자리를 언어가 채운다.** 검출부터 해설까지 전 구간이 비전-언어 모델이다.

| 단 | 모델 | 입력 | 출력 | 라벨 |
|---|---|---|---|---|
| ① 검출 | **OWLv2** (open-vocabulary) | 셀 이미지 + **텍스트 질의**("rust", "torn wrapper" …) | bbox | **0** |
| ② 해설 | **Qwen2.5-VL-7B** (로컬) | 셀 1개의 결함 몽타주 1장 | 종류·이유·심각도·셀 요약 | **0** |

```
셀 1개 = 270 프레임
  → §1 OWLv2 텍스트 질의로 검출        학습 0 · 라벨 0 · 정상셋 0
  → §3 셀 단위로 모아 번호 몽타주 1장
  → §4 로컬 Qwen2.5-VL 1회 호출 → 어떤 결함이고 왜 문제인지
```

**외부 API 호출 없음** — 두 모델 다 Colab GPU에서 돈다(도메인 제약).

⚠️ **평가셋이 아직 없다.** 지금 있는 라벨로 채점하면 8번 반복한 실수를 9번째 반복하는 것이다.
§2는 숫자가 아니라 **육안 관문**이고, 정직한 숫자는 사람이 표시한 150~200장이 생긴 뒤에만 나온다.
판정도 IoU가 아니라 **포함**(보이는 결함이 박스로 덮였는가)이어야 한다 — `0803 §11`이 IoU가 틀린 자임을 이미 보였다.

## 🔴🔴 라벨은 쓰지 않는다

매니페스트 라벨은 **맞는 게 거의 없다**(0804 사용자 판정). 실측 근거:

| 근거 | |
|---|---|
| '무결함'이라던 515·581 | 오염 실재 |
| 표면이 가장 고른 셀 10개를 골라 육안 확인 | **전부 결함** |
| conf≥0.10 검출의 46.4%가 라벨 미대응 | 0731 |
| GT가 결함 전체가 아니라 조각(면적비 p50 112배) | 0803 |

→ 셀의 정상/결함은 **§8에서 눈으로 정한 것만** 쓴다(`§0`의 `CLEAN_OK`/`DEFECT_OK`).
부족해도 라벨로 채우지 않는다 — 추정으로 채우면 또 무효한 판정이 나온다.

**무결함으로 확인된 셀이 아직 0개다** = 정상 클래스가 비어 있어 셀 분류(§6·§7·§9)는 성립하지 않는다.
그 경우 **유일하게 유효한 잣대는 evalset의 사람이 찍은 점 라벨**이고, 그건 정상 셀이 필요 없다.


In [ ]:
# == §0 셋업 — 크롭본 찾기 ==
NB_VER = 'v3'
# 크롭본(357x1024, 셀만 잘린 것)을 쓴다. 배경 86%가 빠져 있어 검출기가 셀에만 집중한다.
import os, glob, zipfile, re
from pathlib import Path
from collections import defaultdict

WORK, SPLITS = Path('/content/work/ext_crop'), ('train', 'val', 'test')
IMG = WORK / 'images'

if not any((IMG / s).exists() for s in SPLITS):
    try:
        from google.colab import drive; drive.mount('/content/drive')
    except Exception: pass
    cand = glob.glob('/content/drive/**/*crop*v42*.zip', recursive=True)
    assert cand, '★ext_crop_v42.zip을 못 찾았다. retrain 노트북 §2로 크롭본을 빌드하거나 Drive 바로가기 확인'
    src = max(cand, key=os.path.getsize)
    print(f'해제: {src} ({os.path.getsize(src)/1e9:.1f} GB)')
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src) as z: z.extractall(WORK)

FILES = [f for s in SPLITS for f in sorted((IMG / s).glob('*.jpg'))]
assert FILES, f'★이미지 0장: {IMG}'

# 파일명 stem에 셀 ID가 있다: EXT__RGB_cell_cylindrical_0921_001__hash
_ID = re.compile(r'cylindrical_(\d+)_')
def cell_id(p):
    m = _ID.search(Path(p).name)
    return int(m.group(1)) if m else -1

BY_CELL = defaultdict(list)
for f in FILES: BY_CELL[cell_id(f)].append(f)
print(f'[{NB_VER}] 크롭본 {len(FILES):,}장 · 셀 {len(BY_CELL):,}개 · 셀당 {len(FILES)/len(BY_CELL):.0f}장')

# 🔴 라벨 의존 금지 — 매니페스트 라벨은 맞는 게 거의 없다.
#    '무결함'이라던 셀에 오염이 실재하고, 표면이 가장 고른 셀 10개도 전부 결함이었다.
# → 아래 목록은 눈으로 본 것만이다. 부족해도 라벨로 채우지 않는다.
CLEAN_OK  = []                 # 육안으로 무결함 확인된 셀 — 0804 현재 **없다**
DEFECT_OK = [515, 581, 534,    # 육안 확인: 결함 있음
             40, 54, 220, 223, 445, 518, 593, 596, 617, 621]   # §9 랭킹 상위 = 전부 결함
N_EACH    = 3                  # 대조에 쓸 정상/결함 셀 수

def pick(confirmed, tag):
    """육안 확인분만. 미확인 셀을 추정으로 채우지 않는다."""
    return [(c, tag) for c in confirmed if c in BY_CELL][:N_EACH]

NEED_BOTH = ('★육안 확인된 정상 셀과 결함 셀이 둘 다 있어야 이 셀이 의미를 가진다.\n'
             '   매니페스트 라벨로는 채우지 않는다 — 맞는 게 거의 없다(0804 실측).\n'
             '   §8로 셀을 직접 보고 §0의 CLEAN_OK / DEFECT_OK에 추가할 것.\n'
             '   🔑 정상 셀을 못 찾으면 이 비교는 아예 성립하지 않는다.\n'
             '      그때 유일하게 유효한 잣대는 **evalset의 사람이 찍은 점 라벨**이다.')

print(f'육안 확인분: 정상 {len(CLEAN_OK)}셀 / 결함 {len(DEFECT_OK)}셀')
if not CLEAN_OK:
    print('⚠️ 정상 셀 0개 — §6·§7·§9(정상 대조)는 못 돈다. 유효한 잣대는 evalset 점 라벨뿐이다.')


In [ ]:
# == §1 검출 모델 — OWLv2 open-vocabulary (텍스트 질의, 라벨 0) ==
# 결함 종류를 텍스트로 지시하므로 학습도 라벨도 정상셋도 필요 없다.
#
# 🔴 OWLv2 함정: 전처리가 이미지를 정사각으로 **패딩**한다. 직사각을 그대로 넣으면
#    반환 좌표가 패딩된 캔버스 기준이라 박스가 어긋난다.
#    → 우리가 먼저 정사각으로 패딩해(원본을 좌상단에) 내부 패딩을 무연산으로 만든다.
!pip -q install -U transformers accelerate

import torch, numpy as np
from PIL import Image
from transformers import Owlv2Processor, Owlv2ForObjectDetection

MODEL_ID = 'google/owlv2-large-patch14-ensemble'   # A100. 느리면 base-patch16-ensemble
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
proc_d = Owlv2Processor.from_pretrained(MODEL_ID)
det = Owlv2ForObjectDetection.from_pretrained(MODEL_ID).to(DEV).eval()

# 질의어 = 이 파이프라인의 실질 "라벨". 설계 규칙 (OWLv2는 CLIP 계열 텍스트 인코더다)
#   ① 영어 · 짧은 명사구. 문장이나 형용사 나열은 신호가 희석된다
#   ② 추상어("defect", "anomaly")는 안 먹는다 — **눈에 보이는 물체/현상**으로 쓴다
#   ③ 한 한국어 유형에 영어 표현 여러 개를 붙여 **OR**로 잡는다(표현마다 반응이 다르다)
#   ④ 한국어 키 = evalset §B 라벨링 UI의 유형과 같은 이름 → §C가 유형별 recall을 낸다
QUERY_MAP = {
    '녹·부식':     ['rust', 'orange brown discoloration', 'brown stain on surface',
                    'oxidation on surface', 'reddish brown spot'],
    '벗겨짐·박리': ['peeling plastic wrap', 'peeled off label', 'exposed metal under wrapper',
                    'wrapper coming off'],
    '파손·찢김':   ['tear in plastic film', 'crack on surface', 'puncture hole', 'dent'],
    '긁힘·스크래치': ['scratch', 'scratch mark', 'scratched metal surface', 'scuff mark'],
    '들뜸':        ['air bubble under film', 'wrinkled plastic wrap', 'lifted film edge'],
    '오염·이물질': ['dirt stain', 'smudge', 'white residue', 'foreign particle on surface'],
}
# 방해 질의(distractor) — OWLv2는 박스마다 가장 잘 맞는 질의를 고른다. 정상 요소에
# 이름을 주면 그 박스가 결함 질의를 훔쳐가지 않고 제 이름으로 붙는다 = 걸러낼 수 있다.
NEG_MAP = {
    '정상:인쇄·각인': ['printed text', 'printed letters on label', 'engraved serial number'],
    '정상:반사':      ['light reflection', 'specular highlight'],
    '정상:금속캡':    ['metal cap', 'battery terminal'],
    '정상:테두리':    ['bottom edge of cylinder', 'rim of metal can', 'silhouette edge'],
}
# 🔴 버릴 것은 셋뿐이다. 금속캡은 절대 안 버린다 — 구조물 판정으로 지우면
#    오탐 5개를 지우고 진짜 결함 9개를 잃는다(순손실).
#    캡 위의 부식은 결함이다. 이름만 붙이고 남긴다.
DROP_TAGS = {'정상:인쇄·각인', '정상:반사', '정상:테두리'}

QUERIES = [q for v in QUERY_MAP.values() for q in v] + \
          [q for v in NEG_MAP.values() for q in v]
TAG_OF  = {q: k for k, v in {**QUERY_MAP, **NEG_MAP}.items() for q in v}

THR   = 0.10   # ★교정 노브. 낮추면 후보↑. 아래 질의별 분포를 보고 조정
TOPK  = 12     # 장당 박스 상한
TILES = 1      # ★1=크롭 통짜. 작은 결함을 놓치면 3(세로 3등분)으로 — 패딩 손실도 같이 준다

# 🔴 `getattr(x, 'a', x.b)`는 **기본값을 먼저 평가**한다 → `x.b`가 없으면 폴백 전에 죽는다.
#    지연 평가(or)로 있는 쪽을 고른다.
_POST = (getattr(proc_d, 'post_process_grounded_object_detection', None)
         or getattr(proc_d, 'post_process_object_detection', None))
assert _POST, f'★후처리 API를 못 찾음: {[a for a in dir(proc_d) if "post_process" in a]}'
print(f'후처리 API: {_POST.__name__}')

@torch.inference_mode()
def detect(paths, thr=THR, topk=TOPK, tiles=TILES, bs=8, drop_neg=True):
    """→ {path: [(x1,y1,x2,y2,score,tag,query), ...]}  원본 크롭 좌표계
       tag = 한국어 유형(§B와 동일) 또는 '정상:...'.  drop_neg=False면 정상 태그도 남긴다"""
    jobs = []
    for p in paths:
        im = Image.open(p).convert('RGB')
        h = im.height // tiles
        for t in range(tiles):
            y0 = t * h
            sub = im.crop((0, y0, im.width, im.height if t == tiles - 1 else y0 + h))
            S = max(sub.size)                    # ★우리가 먼저 정사각 패딩 → 내부 패딩 무연산
            sq = Image.new('RGB', (S, S), (0, 0, 0)); sq.paste(sub, (0, 0))
            jobs.append((p, sq, 0, y0, S))
    out = {p: [] for p in paths}
    for i in range(0, len(jobs), bs):
        blk = jobs[i:i + bs]
        inp = proc_d(text=[QUERIES] * len(blk), images=[j[1] for j in blk],
                     return_tensors='pt').to(DEV)
        o = det(**inp)
        sizes = torch.tensor([[j[4], j[4]] for j in blk], device=DEV)   # 패딩 후 정사각 크기
        for (p, _sq, xo, yo, _S), r in zip(blk, _POST(outputs=o, threshold=thr,
                                                      target_sizes=sizes)):
            for b, s, l in zip(r['boxes'].tolist(), r['scores'].tolist(),
                               r['labels'].tolist()):
                q = QUERIES[l]; tag = TAG_OF[q]
                if drop_neg and tag in DROP_TAGS: continue
                out[p].append((b[0] + xo, b[1] + yo, b[2] + xo, b[3] + yo,
                               float(s), tag, q))
    for p in out:
        out[p].sort(key=lambda t: -t[4]); out[p] = out[p][:topk]
    return out

# ── 질의별 점수 분포 — THR을 감이 아니라 실측으로 정한다 ──
# 🔴 질의마다 점수 스케일이 다르다. 전역 THR 하나면 잘 반응하는 질의 하나가 다 먹는다.
#    아래 표에서 한 질의만 잡히면 그 질의를 빼거나 다른 표현으로 바꿀 것.
import random, inspect
from collections import Counter
assert 'drop_neg' in inspect.signature(detect).parameters, '★옛 detect가 살아 있다 — 이 셀 전체를 붙여넣었는지, 아래에 옛 §1 셀이 또 있는지 확인'
_s = random.Random(42).sample(FILES, min(16, len(FILES))) if 'FILES' in dir() else []
_r = detect(_s, thr=0.01, drop_neg=False) if _s else {}
_all = [b for v in _r.values() for b in v]
if not _s:
    print('⏭️ §0 미실행 — 질의별 점수 분포는 건너뛴다. detect()는 그대로 쓸 수 있다')
elif _all:
    _sc = np.array([b[4] for b in _all])
    print(f'표본 {len(_s)}장 · 후보 {len(_all)}개 · 점수 p50 {np.median(_sc):.3f} '
          f'p90 {np.percentile(_sc,90):.3f} max {_sc.max():.3f}\n')
    print(f'{"질의":38s} {"n":>5s} {"p50":>6s} {"max":>6s}   유형')
    for q in QUERIES:
        v = [b[4] for b in _all if b[6] == q]
        if v: print(f'{q:38s} {len(v):5d} {np.median(v):6.3f} {max(v):6.3f}   {TAG_OF[q]}')
    print('\n유형별:', Counter(b[5] for b in _all).most_common())
    for t in (0.05, 0.10, 0.15, 0.25):
        _k = [b for b in _all if b[4] >= t and b[5] not in DROP_TAGS]
        print(f'  THR {t:.2f} → 장당 {len(_k)/len(_s):5.1f}개 (정상태그 제거 후)')
else:
    print('🔴 후보 0개 — 질의어가 이 도메인에 안 먹는다. QUERY_MAP 표현을 바꾸거나 THR을 더 낮출 것')


In [ ]:
# == §2 육안 관문 — 컨택트 시트 ==
# 판정은 숫자가 아니라 눈이다. "육안으로 보이는 녹·벗겨짐·파손에 박스가 붙는가?"
# 회색 = 정상 태그(인쇄·반사)로 걸러진 것. **걸러진 게 진짜 결함이면 DROP_TAGS를 줄일 것.**
import random
from PIL import Image, ImageDraw
from collections import Counter

N, COLS, TW, TH = 24, 8, 180, 512
sel = random.Random(42).sample(FILES, min(N, len(FILES)))
res = detect(sel, drop_neg=False)          # 걸러지는 것도 봐야 정책을 검증한다

assert all(len(b) == 7 for v in res.values() for b in v), \
    '★detect가 7튜플을 안 뱉는다 = 옛 §1이 도는 중. §1을 새 버전으로 갈 것'
tiles_, kept, dropped = [], 0, 0
for p in sel:
    im = Image.open(p).convert('RGB'); dr = ImageDraw.Draw(im)
    for x1, y1, x2, y2, s, tag, q in res[p]:
        drop = tag in DROP_TAGS
        dropped += drop; kept += not drop
        dr.rectangle([x1, y1, x2, y2],
                     outline=(120, 120, 120) if drop else (255, 40, 40),
                     width=3 if drop else 4)
        dr.text((x1 + 3, max(0, y1 - 12)), f'{tag.split(":")[-1][:6]} {s:.2f}',
                fill=(200, 200, 200) if drop else (255, 255, 0))
    tiles_.append(im.resize((TW, TH)))

rows = (len(tiles_) + COLS - 1) // COLS
sheet = Image.new('RGB', (COLS * TW, rows * TH), 'white')
for i, t in enumerate(tiles_): sheet.paste(t, (i % COLS * TW, i // COLS * TH))
sheet.save('/content/ext_owlv2_sheet.jpg', quality=90)
print(f'{len(sel)}장 · 빨강(결함후보) {kept}개 = 장당 {kept/len(sel):.1f} · 회색(정상태그) {dropped}개')
print('유형별:', Counter(b[5] for v in res.values() for b in v).most_common())
try:
    from IPython.display import display; display(sheet)
except Exception: pass


In [ ]:
# == §3 셀 단위 집계 → 번호 몽타주 1장 ==
# 셀 270장에서 상위 박스만 모아 몽타주 1장 → VLM 호출 1회 (SoM 변형).
from PIL import Image, ImageDraw, ImageFont
from collections import Counter

CELL_TOPK  = 9     # 3x3. 늘리면 칸이 작아져 화질이 떨어진다
PAD        = 24    # 크롭 여백 px — 문맥이 있어야 VLM이 판단한다
MIN_CROP   = 128   # ★박스가 작아도 최소 이만큼은 잘라 온다. EXT 결함은 중앙값 15~25px(0731 §1)라
                   #   PAD만으로는 60px대 조각이 되고, VLM이 "무늬 없는 표면"이라 답한다
TILE       = 320
FRAMES_CAP = 40    # 셀 270장 중 균등 샘플. None이면 전량(느림)

PER_TAG_CAP   = 2   # 한 유형이 몽타주에서 차지할 수 있는 최대 칸. None이면 무제한(구 동작)
PER_FRAME_CAP = 1   # 같은 프레임에서 여러 칸 뽑지 않는다(270장이나 있다)

def cell_report(cid, frames_cap=FRAMES_CAP, per_tag=PER_TAG_CAP, per_frame=PER_FRAME_CAP):
    fs = sorted(BY_CELL[cid])
    if frames_cap: fs = fs[::max(1, len(fs) // frames_cap)][:frames_cap]
    res = detect(fs)                                    # 정상 태그는 이미 걸러진 상태
    pool = sorted(((s, tag, q, p, (x1, y1, x2, y2))
                   for p, v in res.items() for x1, y1, x2, y2, s, tag, q in v),
                  key=lambda t: -t[0])
    cand, ntag, nfr = [], Counter(), Counter()
    for strict in (True, False):             # 1차: 쿼터 지킴 / 2차: 자리가 남으면 채움
        for c in pool:
            if len(cand) >= CELL_TOPK: break
            if c in cand: continue
            if strict and per_tag   and ntag[c[1]] >= per_tag:   continue
            if strict and per_frame and nfr[c[3]] >= per_frame:  continue
            cand.append(c); ntag[c[1]] += 1; nfr[c[3]] += 1
        if len(cand) >= CELL_TOPK: break
    print(f'  [셀 {cid}] 유형 분포 {dict(ntag)}  (후보 풀 {len(pool)}개)')
    items = []
    for s, tag, q, p, (x1, y1, x2, y2) in cand:
        im = Image.open(p).convert('RGB')
        # PAD를 준 뒤에도 MIN_CROP보다 작으면 중심을 기준으로 넓힌다(가장자리에서는 안쪽으로 민다)
        bx = [x1 - PAD, y1 - PAD, x2 + PAD, y2 + PAD]
        for ax, lim in ((0, im.width), (1, im.height)):
            lo, hi = bx[ax], bx[ax + 2]
            if hi - lo < MIN_CROP:
                c = (lo + hi) / 2
                lo, hi = c - MIN_CROP / 2, c + MIN_CROP / 2
            lo, hi = max(0, lo), min(lim, hi)
            if hi - lo < MIN_CROP:                      # 이미지가 더 좁으면 전체를 쓴다
                lo, hi = (0, lim) if lim < MIN_CROP else \
                         ((0, MIN_CROP) if lo == 0 else (lim - MIN_CROP, lim))
            bx[ax], bx[ax + 2] = lo, hi
        # 정사각으로 맞춘다 — 길쭉한 크롭은 몽타주 칸에서 좌우가 흰 여백이 된다(실측 확인)
        cx, cy = (bx[0] + bx[2]) / 2, (bx[1] + bx[3]) / 2
        half = max(bx[2] - bx[0], bx[3] - bx[1], MIN_CROP) / 2
        half = min(half, im.width / 2, im.height / 2)
        cx = min(max(cx, half), im.width - half); cy = min(max(cy, half), im.height - half)
        bx = [cx - half, cy - half, cx + half, cy + half]
        items.append({'crop': im.crop(tuple(map(int, bx))),
                      'tag': tag, 'query': q, 'score': s, 'frame': Path(p).name})
    return items, len(fs)

def _font(sz):
    for p in ('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
              '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
              '/usr/share/fonts/truetype/freefont/FreeSansBold.ttf',
              'C:/Windows/Fonts/arialbd.ttf'):
        try: return ImageFont.truetype(p, sz)
        except Exception: pass
    try: return ImageFont.load_default(size=sz)      # Pillow ≥10.1
    except TypeError: return ImageFont.load_default()

def montage(items, tile=TILE):
    cols = 3 if len(items) > 4 else 2
    rows = max(1, (len(items) + cols - 1) // cols)
    sh = Image.new('RGB', (cols * tile, rows * tile), 'white'); dr = ImageDraw.Draw(sh)
    font = _font(tile // 4)
    _h = dr.textbbox((0, 0), '8', font=font)[3]
    if _h < tile // 8: print(f'  ⚠️ 번호 높이 {_h}px — 폰트 로드 실패. VLM이 번호를 못 읽는다')
    for i, it in enumerate(items):
        # 🔴 `thumbnail()`은 축소만 하고 확대는 안 한다. 작은 크롭이 칸 구석에 작게 붙고
        #    나머지가 흰 여백이 되어 VLM이 사실상 빈 칸을 본다 → resize로 칸을 채운다.
        c = it['crop']
        k = min((tile - 8) / c.width, (tile - 8) / c.height)
        c = c.resize((max(1, round(c.width * k)), max(1, round(c.height * k))), Image.LANCZOS)
        ox, oy = i % cols * tile, i // cols * tile
        sh.paste(c, (ox + (tile - c.width) // 2, oy + (tile - c.height) // 2))
        dr.rectangle([ox, oy, ox + tile - 1, oy + tile - 1], outline=(0, 0, 0), width=3)
        b = dr.textbbox((0, 0), str(i + 1), font=font)
        w, h = b[2] - b[0] + tile // 12, b[3] - b[1] + tile // 12
        dr.rectangle([ox, oy, ox + w, oy + h], fill=(0, 0, 0))
        dr.text((ox + tile // 24 - b[0], oy + tile // 24 - b[1]), str(i + 1),
                fill=(255, 255, 0), font=font)
    return sh

# ★검사할 셀 목록. 기본은 프레임이 많은 순 3개 — 하나만 보면 그 셀이 대표인지 알 수 없다.
CELL_IDS = sorted(BY_CELL, key=lambda c: -len(BY_CELL[c]))[:3]
print(f'대상 셀 {CELL_IDS} (전체 {len(BY_CELL)}개 중)\n')

CELLS = {}                       # cid → (items, montage, 검사 프레임 수)
for cid in CELL_IDS:
    items, nf = cell_report(cid)
    if not items:
        print(f'셀 {cid}: 후보 0개 — §1의 THR/QUERY_MAP 조정 필요'); continue
    CELLS[cid] = (items, montage(items), nf)
    print(f'셀 {cid} · 프레임 {nf}장 검사 → 후보 {len(items)}개')
    for i, it in enumerate(items, 1):
        print(f'  {i}. {it["tag"]:12s} {it["score"]:.3f}  ({it["query"]})  {it["frame"]}')
    print()
assert CELLS, '★어느 셀에서도 후보가 안 나왔다 — §1을 먼저 조정할 것'

# 🔑 점수를 꼭 볼 것. 전부 THR 근처(0.10~0.12)면 후보가 노이즈다 = VLM이 "정상"이라 답하는 게 정상.
CID, (ITEMS, MONT, _nf) = next(iter(CELLS.items()))     # §4 기본 대상 = 첫 셀
MONT.save('/content/ext_cell_montage.jpg', quality=92)
try:
    from IPython.display import display
    for cid, (_it, mo, _n) in CELLS.items(): print(f'── 셀 {cid} ──'); display(mo)
except Exception: pass


In [ ]:
# == §4 로컬 VLM — 어떤 결함이고 왜 그런가 (외부 API 없음) ==
# 도메인 제약상 외부 API 금지 → 로컬 Qwen2.5-VL-7B. A100 40GB에 bf16 ~16GB.
# 몽타주 1장 = 셀 1개 = 호출 1회. 프레임 수(270)와 무관하게 지연이 상수다.
!pip -q install -U "transformers>=4.49" accelerate qwen-vl-utils

import torch, json, re
from collections import Counter
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

VLM_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'
vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_ID, torch_dtype=torch.bfloat16, device_map='auto')
proc_v = AutoProcessor.from_pretrained(VLM_ID)

# 심각도 5등급 — 리튬셀 안전 기준. 상/중/하 3단계는 전부 한 값으로 뭉쳐 변별력이 0이었다.
# 진짜 분기점은 크기가 아니라 금속이 드러났는가 / 안전변이 성한가다.
# 1등급은 "결함인지도 애매한 것"에만 준다 — 여기가 헐거우면 전부 1등급으로 샌다.
GRADES = """5 치명 : 금속이 드러남(외피 관통·찢김), 캡·안전변 변형·파손, 깊게 눌림  → 폐기
4 중대 : 외피 벗겨짐·박리가 진행 중, 넓은 부식(사진 면적의 1/4 이상)      → 폐기 후보
3 보통 : 뚜렷한 국소 부식·변색·외피 들뜸. **사진을 열자마자 눈에 띄는 크기** → 재검사
2 경미 : 자세히 봐야 보이는 얕은 긁힘·옅은 얼룩. 폭이 가늘고 국소적       → 기록만
1 무시 : 머리카락처럼 가는 선, 확대해야 겨우 보이는 점·얼룩              → 통과

⚠️ **3등급 이상은 사진을 보자마자 바로 눈에 띄는 것에만 준다.**
   자세히 들여다봐야 겨우 보이면 1~2등급이다. 가는 선 하나는 1등급이다."""

# 종류는 **evalset §B TYPES와 같은 어휘**로 강제한다 — 안 그러면 유형별 recall이 안 붙는다.
KIND_TXT = ' | '.join(list(QUERY_MAP) + ['기타'])
NORM_TXT = ' | '.join(['각인·인쇄', '반사·얼룩', '정상 표면'])

def build_prompt(items, solo=False):
    # 검출기의 추측을 같이 넘긴다 — 다만 "참고"로만. 그대로 베끼면 VLM을 쓰는 의미가 없다.
    if solo:
        head = (f"""이 이미지는 원통형 리튬 배터리 셀 표면에서 검출기가 이상하다고 표시한 부위를
확대한 것이다. 검출기 추측 = {items[0]['tag']} (틀릴 수 있다. 네 눈으로 다시 판단하라).

이 부위 하나를 판정하라.""")
    else:
        hint = '\n'.join(f'  {i}번: 검출기 추측 = {it["tag"]}' for i, it in enumerate(items, 1))
        head = (f"""이 이미지는 원통형 리튬 배터리 셀 **한 개**를 360도 회전시켜 촬영한 사진에서
검출기가 이상하다고 표시한 부위들을 잘라 번호를 붙여 격자로 붙인 것이다.

검출기의 추측(틀릴 수 있다. 참고만 하고 네 눈으로 다시 판단하라):
{hint}

**{len(items)}개 칸을 하나씩 따로** 보고 판정하라. 칸마다 사진이 다르므로
🔴 **판정·등급·근거가 칸마다 달라야 정상이다.** 한 칸을 보고 나머지에 같은 답을 복사하지 마라.
근거에는 **그 칸에서만 보이는 것**을 써라(색·위치·형태).""")

    tail = '' if solo else (
        '\n마지막에 이 셀 전체에 대한 한 문단 요약'
        '(어떤 결함이 어디에 몇 개 있고 왜 문제인지)을 쓰라.\n')
    summ = '' if solo else ',"셀요약":"<한 문단>"'
    no = '' if solo else '"번호":<1부터>,'
    return f"""{head}
- 관찰: **먼저** 이 사진에 실제로 보이는 것만 한 문장으로 적는다(판단 없이 묘사만)
- 판정: 결함 / 정상
- 종류: 결함이면 [{KIND_TXT}] 중 하나, 정상이면 [{NORM_TXT}] 중 하나
- 위치: 금속캡 | 상단 | 몸통 | 하단 | 불명
- 등급: 아래 1~5 중 하나 (숫자만)
{GRADES}
- 근거: 관찰한 것 중 무엇 때문에 그렇게 판정했는지 한 문장

판정 기준:
- 외피(비닐 수축튜브)가 찢어져 **속의 흰색·은색이 드러났으면** 결함(등급 4~5).
- 외피가 벗겨지거나 들뜬 것도 결함이다. **아주 미약해도 결함**이며 등급 2~3으로 낮게 주되
  '정상'으로 넘기지 마라.
- 표면에 갈색 부식·변색·이물질이 있으면 결함이다.
- 각인·인쇄(제조번호·모델명 문자), 빛 반사, 균일한 무늬 없는 표면은 '정상'이다.
- **가늘고 검은 선·줄은 '긁힘·스크래치'다** — 부식이나 오염이 아니다. 같은 것에는 같은 이름을 쓴다.
- 🔴 **사진에 금속캡(원형 단자면)이 안 보이면 위치에 '금속캡'을 쓰지 마라.**
  옆면만 보이면 '몸통'이다. 보이지 않는 것을 있다고 적지 마라.
- 🔴 **검사 대상에는 아무 이상 없는 정상 셀도 섞여 있다.** 검출기가 표시했다는 이유만으로
  결함이라고 하지 마라. 이상이 안 보이면 주저 없이 '정상'으로 판정하라.
  실제로 여기 오는 사진의 상당수는 정상이다.
- 결함이라고 할 때는 **무엇이 어떻게 이상한지 관찰에 적을 수 있어야** 한다.
  "표면이 균일하다", "색이 고르다"면 그건 정상이다.
{tail}
출력은 JSON만. **아래는 예시가 아니라 빈칸이다** — `< >` 안의 지시대로
네가 실제로 본 것으로 채워라. `< >` 안의 글자를 그대로 옮겨 적지 마라:
{{"항목":[{{{no}"관찰":"<보이는 것 묘사>","판정":"<결함 또는 정상>","종류":"<위 목록에서 하나>",\
"위치":"<위 목록에서 하나>","등급":<1~5 정수>,"근거":"<한 문장>"}}]{summ}}}"""

def _gen(img, prompt, max_new=1600):
    msgs = [{'role': 'user', 'content': [{'type': 'image', 'image': img},
                                         {'type': 'text', 'text': prompt}]}]
    text = proc_v.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = proc_v(text=[text], images=[img], return_tensors='pt').to(vlm.device)
    with torch.inference_mode():
        gen = vlm.generate(**inp, max_new_tokens=max_new, do_sample=False)
    ans = proc_v.batch_decode(gen[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0]
    m = re.search(r'\{.*\}', ans, re.S)
    try: return (json.loads(m.group(0)) if m else {'항목': [], '셀요약': ans})
    except json.JSONDecodeError: return {'항목': [], '셀요약': f'[JSON 파싱 실패] {ans[:400]}'}

# → 기본은 per_item. 칸당 크롭 1장씩 따로 물으면 번호↔칸 매핑 문제가 원천적으로 없다.
#    비용 셀당 ~9회 호출(~20초). montage는 검증용으로만 남긴다.
VLM_MODE = 'per_item'

def ask_vlm(mont, items, mode=None):
    mode = mode or VLM_MODE
    if mode == 'montage':
        return _gen(mont, build_prompt(items))
    out = []
    for i, it in enumerate(items, 1):                     # per_item: 칸마다 원본 크롭으로 1회
        r = _gen(it['crop'], build_prompt([it], solo=True), max_new=400)
        d = (r.get('항목') or [{}])[0]; d['번호'] = i; out.append(d)
    return {'항목': out, '셀요약': '(per_item 모드 — 셀 요약 없음)'}

def show_cell(cid, display_montage=True, mode=None):
    """🔑 결함만이 아니라 **전 항목**을 찍는다 — 결함 0개일 때 아무것도 안 보이면
       VLM이 무엇을 보고 그렇게 판단했는지 검증할 수 없다(초판의 설계 결함)."""
    items, mont, nf = CELLS[cid]
    rep = ask_vlm(mont, items, mode)
    by_no = {int(r['번호']): r for r in rep.get('항목', []) if str(r.get('번호', '')).isdigit()}
    n_def = sum(1 for r in by_no.values() if r.get('판정') == '결함')
    print(f'\n{"="*70}\n■ 셀 {cid} · 프레임 {nf}장 · 후보 {len(items)}개 → 결함 {n_def}개')
    if display_montage:
        try:
            from IPython.display import display; display(mont)
        except Exception: pass
    for i, it in enumerate(items, 1):
        r = by_no.get(i)
        if r is None:
            print(f'  {i}. ⚠️ VLM 무응답        | 검출기 {it["tag"]} {it["score"]:.3f}'); continue
        mark = '🔴결함' if r.get('판정') == '결함' else '  정상'
        agree = '=' if r.get('종류') == it['tag'] else '≠'
        print(f'  {i}. {mark} [{r.get("등급")}등급] {str(r.get("종류")):12s} '
              f'@{str(r.get("위치", "?")):5s} {agree} 검출기 {it["tag"]} {it["score"]:.3f}')
        if r.get('관찰'): print(f'      관찰: {r["관찰"]}')
        print(f'      근거: {r.get("근거")}')
        print(f'      {it["frame"]}')

    vals = list(by_no.values())
    reasons = [str(r.get('근거', '')) for r in vals]
    if len(reasons) >= 4 and len(set(reasons)) <= max(1, len(reasons) // 4):
        print(f'\n  ⚠️ 근거 {len(reasons)}개 중 서로 다른 것이 {len(set(reasons))}개뿐 '
              f'= 사진별로 안 보고 복사한 것')
    # 템플릿 복사 — 프롬프트 예시값(결함·녹·부식·금속캡·3등급)이 그대로 나오면 사진을 안 본 것
    if len(vals) >= 4:
        sig = [(r.get('판정'), r.get('종류'), r.get('위치'), r.get('등급')) for r in vals]
        top, n = Counter(sig).most_common(1)[0]
        if n / len(sig) >= 0.7:
            print(f'\n  🔴 {n}/{len(sig)}이 완전히 같은 조합 {top} = **프롬프트 예시를 베끼는 중**.'
                  f'\n     JSON 스키마의 값을 <플레이스홀더>로 바꿨는지 확인할 것')
    print(f'\n[셀 요약] {rep.get("셀요약", "")}')
    return rep

print('§4 준비 완료 — _gen / ask_vlm / show_cell 정의됨. '
      '셀 리포트는 §4-run, 평가셋 전량 덤프는 evalset §E')


In [ ]:
# == §4-run 셀 리포트 실행 (선택 — 평가셋 작업에는 필요 없다) ==
# ⚠️ 이 경로는 몽타주 시절 프롬프트를 그대로 쓴다 — JSON 예시가 들어 있어 모델이
#    예시를 베끼는 경로다. 숫자가 필요하면 evalset §E → §F로 간다.
assert CELLS, '★§3을 먼저 (CELLS가 없다)'
REPORTS = {cid: show_cell(cid) for cid in CELLS}



In [ ]:
# == §5 (선택) 프레임 프리필터 — 지연 축소 ==
# 셀 1개 = 270 프레임을 전부 OWLv2에 넣으면 느리다. 값싼 색도/텍스처 맵으로 상위 K장만 골라 넘긴다.
# 정확도용이 아니라 **지연용**이다. §3의 FRAMES_CAP(균등 샘플)을 이걸로 바꾸면 결함 프레임을 우선한다.
#
# 채널 2개 (합성 픽스처로 검증한 사실)
#   rust    : Lab 색도에서 "유채색이면서 색상이 다른" 것만 → 녹·부식·변색
#   texture : L채널 국소 표준편차 → 벗겨짐·찢김·파손·들뜸
import numpy as np, tempfile, os
from PIL import Image
from scipy import ndimage

def _lab(rgb):
    m = np.array([[0.412, 0.358, 0.180], [0.213, 0.715, 0.072], [0.019, 0.119, 0.950]])
    xyz = rgb @ m.T / np.array([0.9505, 1.0, 1.089])
    f = np.where(xyz > 0.008856, np.cbrt(xyz), 7.787 * xyz + 16 / 116)
    return 116 * f[..., 1] - 16, 500 * (f[..., 0] - f[..., 1]), 200 * (f[..., 1] - f[..., 2])

CHROMA_THR, MIN_AREA = 10.0, 300     # ★교정 노브 (Lab 색도 단위 / px^2)

def _peak(z, thr, min_area=MIN_AREA):
    """면적을 통과한 덩어리의 최대값. 🔴이 필터가 이 채널의 판별력 자체다 —
       없이 그냥 max를 쓰면 각인의 JPEG 경계 헤일로가 녹만큼 찍힌다(실측 11.07 vs 35.38)."""
    m = ndimage.binary_opening(z > thr, np.ones((5, 5)))
    lab_, _ = ndimage.label(m)
    best = 0.0
    for sl in ndimage.find_objects(lab_):
        y, x = sl
        if (y.stop - y.start) * (x.stop - x.start) >= min_area:
            best = max(best, float(z[sl].max()))
    return best

def anomaly_score(path):
    """→ (색도 이탈, 텍스처 이탈). 셀 자신이 참조라 정상셋이 필요 없다"""
    rgb = np.asarray(Image.open(path).convert('RGB'), np.float32) / 255.0
    L, a, b = _lab(rgb)
    ab = np.stack([a, b], -1)
    ref = np.median(ab, axis=0, keepdims=True)          # 열별 참조 = 원통 곡면 음영 흡수
    C, Cr = np.linalg.norm(ab, axis=-1), np.linalg.norm(ref, axis=-1)
    cos = (ab * ref).sum(-1) / (C * Cr + 1e-6)
    rust = np.minimum(C, Cr) * (1 - cos) / 2            # 무채색이면 min(C,Cr)~0 → 조용
    m1 = ndimage.uniform_filter(L, 9)
    tex = np.sqrt(np.maximum(ndimage.uniform_filter(L * L, 9) - m1 * m1, 0))
    tex = tex - np.median(tex, axis=0, keepdims=True)
    med = np.median(tex); mad = max(np.median(np.abs(tex - med)), 0.5)   # ★MAD 바닥
    return _peak(rust, CHROMA_THR), _peak(tex, med + 6 * 1.4826 * mad)

# ── 자체검증: 합성 셀 (녹은 잡고 무채색 인쇄/각인은 안 잡아야 한다) ──
_rng = np.random.default_rng(0)
_a = np.zeros((1024, 357, 3), np.float32); _a[:, :] = (40, 70, 160)
_a += np.linspace(-25, 25, 357)[None, :, None] + _rng.normal(0, 3, _a.shape)
_a[300:340, 60:300] = 245          # 흰 인쇄 문구 (무채색)
_a[120:150, 40:70] = 10            # 검은 각인   (무채색)
_p = os.path.join(tempfile.gettempdir(), '_synth.jpg')
Image.fromarray(_a.clip(0, 255).astype(np.uint8)).save(_p, quality=95)
_c_rust, _c_tex = anomaly_score(_p)
_a[600:700, 150:260] = (150, 90, 35)                  # 갈색 녹 추가
Image.fromarray(_a.clip(0, 255).astype(np.uint8)).save(_p, quality=95)
_r_rust, _r_tex = anomaly_score(_p)
assert _c_rust < 1.0,  f'★무채색(인쇄·각인)이 색도 채널을 울린다: {_c_rust:.2f}'
assert _r_rust > 10.0, f'★녹을 못 잡는다: {_r_rust:.2f} — CHROMA_THR을 낮출 것'
print(f'PASS — 색도: 무채색만 {_c_rust:.2f} → 녹 추가 {_r_rust:.2f}')
print(f'       텍스처: {_c_tex:.1f} → {_r_tex:.1f} (인쇄·각인에 반응하는 게 정상, §4가 거른다)')

def top_frames(cid, k=40):
    """셀에서 이상 점수 상위 k 프레임. §3의 균등 샘플을 대체한다"""
    fs = sorted(BY_CELL[cid])
    sc = [(max(anomaly_score(p)), p) for p in fs]
    sc.sort(key=lambda t: -t[0])
    return [p for _s, p in sc[:k]]
print(f'\n사용법:  §3의 cell_report에서 fs = top_frames(cid, 40) 로 교체')


In [ ]:
# == §6 정상 셀 대조 — 오탐률 측정 ==
# 🔴 정상/결함 라벨은 §0의 육안 확인분만 쓴다. 매니페스트로는 채우지 않는다.
from collections import Counter
assert 'pick' in dir(), '★§0을 먼저 실행하세요 (라벨 블록이 §0에 있습니다)'

CMP_FRAMES = 20  # 셀당 검사 프레임. §4의 40보다 줄여 시간을 절반으로

TEST = pick(CLEAN_OK, '정상') + pick(DEFECT_OK, '결함')
assert CLEAN_OK and DEFECT_OK, NEED_BOTH
print('육안 확인분: 정상 {} / 결함 {}'.format(CLEAN_OK, DEFECT_OK))
if any(k.endswith('?') for _, k in TEST):
    print("⚠️ '?' 붙은 셀은 **매니페스트 추정**이다 — 오라벨일 수 있으니 §8로 확인 후 위 목록에 옮길 것")
print(f'대조 {len(TEST)}셀: ' + ', '.join(f'{c}({k})' for c, k in TEST) + '\n')

ROWS = []
for cid, kind in TEST:
    items, nf = cell_report(cid, frames_cap=CMP_FRAMES)
    if not items:
        ROWS.append((cid, kind, 0, [], 0)); print(f'  셀 {cid}: 후보 0개'); continue
    rep = ask_vlm(montage(items), items)
    gr = [int(r['등급']) for r in rep.get('항목', [])
          if r.get('판정') == '결함' and str(r.get('등급', '')).isdigit()]
    ROWS.append((cid, kind, len(items), gr, max(gr) if gr else 0))
    print(f'  셀 {cid} [{kind}] 후보 {len(items)} → 결함 {len(gr)} '
          f'등급 {sorted(gr, reverse=True)}')

print(f'\n{"셀":>6s} {"구분":14s} {"후보":>4s} {"결함":>4s} {"최고등급":>6s}')
for cid, kind, n, gr, mx in ROWS:
    print(f'{cid:>6d} {kind:14s} {n:>4d} {len(gr):>4d} {mx:>6d}')

# ── 셀 단위 판정: "최고 등급 ≥ T 이면 불량" — T를 훑는다 ──
# 리포트가 셀 하나를 통과/재검으로 가르는 지점이 여기다. 제품 운영점 선택용.
print(f'\n{"임계 T":>6s} {"정상셀 오탐":>10s} {"결함셀 검출":>10s}   판정')
for T in (1, 2, 3, 4, 5):
    cl = [r for r in ROWS if r[1].startswith('정상')]
    df = [r for r in ROWS if r[1].startswith('결함')]
    fp = sum(1 for r in cl if r[4] >= T); tp = sum(1 for r in df if r[4] >= T)
    print(f'{T:>6d} {fp:>4d}/{len(cl):<5d} {tp:>4d}/{len(df):<5d}   '
          + ('✅ 분리' if fp == 0 and tp == len(df) else
             '오탐 0이나 놓침 있음' if fp == 0 else
             '전부 불량 처리' if tp == len(df) and fp == len(cl) else '혼재'))

print("""
읽는 법
  · 정상(같은배치) 셀에 결함이 뜬다        → 진짜 오탐. 등급 기준을 올리거나 THR을 올린다
  · 정상(다른배치)만 깨끗하다              → 배치 차이를 본 것일 수 있다. 같은배치 결과가 정답에 가깝다
  · 어떤 T에서도 분리가 안 된다            → 등급이 변별력을 못 낸다. §4 GRADES 재조정
  · 정상 셀 후보 수가 결함 셀과 비슷하다   → 검출기는 원래 다 찍는다. 거르는 건 VLM 몫이라 정상 동작""")


In [ ]:
# == §7 THR 교정 — 검출기에 분리력이 있는지부터 잰다 (VLM 안 씀) ==
# 답할 질문 하나: OWLv2 점수가 정상 셀과 결함 셀을 가르기는 하는가?
import numpy as np, random
from collections import defaultdict

CAL_FRAMES = 20
CAL_N      = 3      # 정상/결함 각 몇 셀
assert 'pick' in dir(), '★§0을 먼저 실행하세요 (확인분 목록이 §0에 있습니다)'
# 🔴 §0의 육안 확인분만 쓴다. 매니페스트로 잰 이전 판정은 무효였다.
assert CLEAN_OK and DEFECT_OK, NEED_BOTH
CAL = pick(CLEAN_OK, '정상')[:CAL_N] + pick(DEFECT_OK, '결함')[:CAL_N]
print('육안 확인분: 정상 {} / 결함 {}'.format(CLEAN_OK, DEFECT_OK))

SC = {}
for cid, kind in CAL:
    fs = sorted(BY_CELL[cid])
    fs = fs[::max(1, len(fs) // CAL_FRAMES)][:CAL_FRAMES]
    r = detect(fs, thr=0.01, topk=10**6)          # ★상한 없이 전부 — 상한이 있으면 포화만 본다
    SC[cid] = (kind, np.array([b[4] for v in r.values() for b in v]), len(fs))
    s = SC[cid][1]
    print(f'  셀 {cid:>5d} [{kind:9s}] {len(fs)}장 · 검출 {len(s):>5d} '
          f'({len(s)/len(fs):5.1f}/장) · max {s.max() if len(s) else 0:.3f} '
          f'· p95 {np.percentile(s,95) if len(s) else 0:.3f}')

# ── THR별 장당 검출 수: 정상 vs 결함 ──
print(f'\n{"THR":>6s} {"정상/장":>8s} {"결함/장":>8s} {"배율":>7s}  판정')
best = None
for t in (0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60):
    per = defaultdict(list)
    for cid, (kind, s, nf) in SC.items():
        per['정상' if kind.startswith('정상') else '결함'].append((s >= t).sum() / nf)
    cl = np.mean(per['정상']) if per['정상'] else 0
    df = np.mean(per['결함']) if per['결함'] else 0
    ratio = df / cl if cl > 0.01 else (float('inf') if df > 0.01 else 1.0)
    tag = ('✅ 정상 조용 · 결함 검출' if cl < 0.3 and df >= 0.5 else
           '정상도 결함도 조용' if df < 0.5 else
           '⚠️ 정상도 시끄러움')
    print(f'{t:>6.2f} {cl:>8.2f} {df:>8.2f} {ratio:>7.2f}  {tag}')
    if cl < 0.3 and df >= 0.5 and best is None: best = t

# ── 🔑 셀 단위 최고점 — 위 표가 놓치는 신호 ──
# 위는 "장당 몇 개"(밀도)를 봤다. 제품 판정은 **셀 하나를 통과/재검으로 가르는 것**이고,
# 그건 270프레임 중 **가장 강한 검출 하나**로 정해진다. 밀도와 다른 통계다.
_mx = defaultdict(list)
for cid, (kind, s, nf) in SC.items():
    _mx['정상' if kind.startswith('정상') else '결함'].append((s.max() if len(s) else 0, cid, kind))
print('\n■ 셀 최고 점수 (제품 판정에 쓰는 통계)')
for k in ('결함', '정상'):
    print(f'  {k}: ' + ', '.join(f'{c}={v:.3f}' for v, c, _ in sorted(_mx[k], reverse=True)))
_cn = max((v for v, _, _ in _mx['정상']), default=0)
_hit = sum(1 for v, _, _ in _mx['결함'] if v > _cn)
print(f'  → 정상 최고 {_cn:.3f} 초과 = 결함 셀 {_hit}/{len(_mx["결함"])} '
      f'(오탐 0인 지점에서의 셀 검출률)')
if _hit and _hit < len(_mx['결함']):
    print(f'  ⚠️ 밀도로는 안 갈려도 **최고점으로는 부분 분리**된다. 임계 후보 {_cn:.2f}. '
          f'단 셀 {len(SC)}개 표본이라 확정 아님 — CAL_N을 올려 재측정할 것')

print('\n' + '─' * 66)
if best:
    print(f'✅ 추천 THR = {best:.2f}  — §1의 THR을 이 값으로 바꾸고 §3·§6을 다시 돌린다')
    print(f'⚠️ 확인된 셀 {len(CLEAN_OK)}+{len(DEFECT_OK)}개로 정한 값이다. '
          f'§8로 확인분을 늘려 재측정해야 확정된다')
else:
    mx = {k: max(SC[c][1].max() for c in SC if SC[c][0] == k) for k in ('정상', '결함')
          if any(SC[c][0] == k for c in SC)}
    print(f'🔴 어떤 THR에서도 안 갈린다 (정상 최고점 {mx.get("정상", 0):.3f} / '
          f'결함 최고점 {mx.get("결함", 0):.3f})')
    print("""   = OWLv2 점수에 분리력이 없다. THR을 올려봐야 결함도 같이 사라진다.
   다음 수(순서대로):
     ① QUERY_MAP에서 'scratch' 계열 제거 — 어디에나 반응해 점수를 오염시킨다
     ② TILES=3 으로 해상도 확보 후 재측정
     ③ owlv2-large → 다른 open-vocab 검출기, 또는 소량 라벨로 파인튜닝
   ⚠️ 정상 셀 2개(515·581)뿐이라 표본이 작다. 판정 전에 CAL_N을 늘려 재확인할 것""")


In [ ]:
# == §8 🔴 셀을 눈으로 본다 — 유일하게 믿을 수 있는 판정 ==
# 매니페스트 라벨은 못 믿는다 → 셀의 정상/결함은 여기서 눈으로 정하고 §0에 넣는다.
#
# 두 가지를 본다.
#   (A) 전체 프레임 — 셀이 통째로 깨끗한가 (회전 구간을 균등하게)
#   (B) 최고점 크롭 — 검출기가 뭘 보고 점수를 줬나
import numpy as np
from PIL import Image, ImageDraw

VIEW_IDS   = []            # ★확인할 셀. §9 랭킹이 찍어주는 목록을 여기 붙인다
VIEW_DEF   = DEFECT_OK[:1] # 대조용 — 이미 결함으로 확인한 셀(눈 기준점)
assert VIEW_IDS or VIEW_DEF, '★VIEW_IDS에 확인할 셀 ID를 넣으세요 (§9 랭킹 참조)'
N_FRAMES   = 12      # (A) 전체 프레임 몇 장
N_CROPS    = 6       # (B) 최고점 크롭 몇 개
FW, FH     = 150, 430

def sheet_frames(cid, n=N_FRAMES):
    """(A) 셀을 한 바퀴 균등하게 — 통째로 깨끗한지 본다"""
    fs = sorted(BY_CELL[cid]); step = max(1, len(fs) // n); fs = fs[::step][:n]
    sh = Image.new('RGB', (len(fs) * FW, FH), 'white'); dr = ImageDraw.Draw(sh)
    f = _font(26)
    for i, p in enumerate(fs):
        sh.paste(Image.open(p).convert('RGB').resize((FW - 4, FH - 4)), (i * FW + 2, 2))
        dr.rectangle([i * FW, 0, i * FW + 34, 30], fill=(0, 0, 0))
        dr.text((i * FW + 6, 2), str(i + 1), fill=(255, 255, 0), font=f)
    return sh, fs

def sheet_crops(cid, n=N_CROPS):
    """(B) 검출기 최고점 크롭 — 여기 진짜 결함이 보이면 '정상 셀' 라벨이 틀린 것"""
    items, _ = cell_report(cid, frames_cap=20, per_tag=None, per_frame=None)  # 순수 점수순
    items = items[:n]
    if not items: return None, []
    return montage(items), items

for kind, ids in (('미확인', VIEW_IDS), ('결함(확인됨·대조)', VIEW_DEF)):
    for cid in ids:
        print(f'\n{"="*70}\n■ 셀 {cid} — {kind} · 프레임 {len(BY_CELL[cid])}장')
        fr, fs = sheet_frames(cid)
        fr.save(f'/content/cell{cid}_frames.jpg', quality=92)
        cr, its = sheet_crops(cid)
        if cr: cr.save(f'/content/cell{cid}_crops.jpg', quality=92)
        for i, it in enumerate(its, 1):
            print(f'   {i}. {it["tag"]:12s} {it["score"]:.3f}  {it["frame"]}')
        try:
            from IPython.display import display
            print('  (A) 전체 프레임'); display(fr)
            if cr: print('  (B) 최고점 크롭'); display(cr)
        except Exception: pass

print("""
──────────────────────────────────────────────────────────────────
판정 — (B) 크롭에 찢김·벗겨짐·부식이 보이는가?

  🔴 보인다  → 그 셀을 §0의 **DEFECT_OK**에 추가
  ✅ 안 보인다 → 그 셀을 §0의 **CLEAN_OK**에 추가  ← 이게 나와야 §6·§7·§9가 돈다

⚠️ (A)에서 결함이 보이는데 (B)에 안 잡혔다면 그건 **검출기가 놓친 것**이다 —
   그 셀은 결함이고, 별개로 §1 THR을 낮출 근거다. 둘을 섞지 말 것.

🔴🔴 실측(0804): 표면이 가장 고른 셀 10개까지 **전부 결함**이었다.
   무결함으로 확인된 셀이 없다 = **정상 클래스가 비어 있다.**
   → 정상/결함 **셀 분류**는 이 데이터로 성립하지 않는다. 목표를 바꾼다:
     ① 결함 **위치**를 찾는가   → evalset의 사람이 찍은 점 recall. **정상 셀이 필요 없다**
     ② 결함 **심각도**로 가르는가 → 전부 결함이어도 등급 분포는 다르다(폐기/재검/통과)
   ⚠️ "전부 불량"의 기준을 확인할 것 — 머리카락 같은 긁힘 하나면 안전상 1~2등급(통과)이다.
      셀 유무가 아니라 **등급**이 제품 판정이다.""")


In [ ]:
# == §9 표면 균일도 — 셀 단위 스코어 (검출기·VLM 안 씀) ==
!pip -q install scipy
import numpy as np
from PIL import Image
from scipy import ndimage
assert '_lab' in dir(), '★§5를 먼저 실행하세요 (_lab 함수를 씁니다)'

UNI_FRAMES = 20      # 셀당 표본 프레임
L_DEV      = 10.0    # 밝기 이탈 임계(Lab L 단위). 이보다 벗어난 화소를 "얼룩"으로 센다

def surface_stats(path):
    """셀 몸통 안에서만 잰다 → (얼룩비, 색편차비, 질감p99)"""
    rgb = np.asarray(Image.open(path).convert('RGB'), np.float32) / 255.0
    L, a, b = _lab(rgb)
    ab = np.stack([a, b], -1); C = np.linalg.norm(ab, axis=-1)
    # 배경(흰 바탕)은 무채색이라 C가 0에 가깝다 → 유채색 영역만 셀 몸통으로 본다
    m = C > max(5.0, 0.4 * np.percentile(C, 95))
    m = ndimage.binary_opening(m, np.ones((9, 9)))
    if m.sum() < 500: return np.nan, np.nan, np.nan
    Lc = L - np.median(np.where(m, L, np.nan), axis=0, keepdims=True)   # 열별 = 원통 음영 제거
    ref = np.median(ab, axis=0, keepdims=True); Cr = np.linalg.norm(ref, axis=-1)
    cos = (ab * ref).sum(-1) / (C * Cr + 1e-6)
    rust = np.minimum(C, Cr) * (1 - cos) / 2
    m1 = ndimage.uniform_filter(L, 9)
    tex = np.sqrt(np.maximum(ndimage.uniform_filter(L * L, 9) - m1 * m1, 0))
    return ((np.abs(Lc[m]) > L_DEV).mean(),        # 얼룩 — 밝기가 주변과 다른 화소 비율
            (rust[m] > CHROMA_THR).mean(),          # 색편차 — 색상이 다른 화소 비율
            float(np.percentile(tex[m], 99)))       # 질감 — 국소 대비 상위

assert 'pick' in dir(), '★§0을 먼저 실행하세요 (확인분 목록이 §0에 있습니다)'
assert CLEAN_OK and DEFECT_OK, NEED_BOTH
TARGETS = pick(CLEAN_OK, '정상') + pick(DEFECT_OK, '결함')

print(f'{"셀":>6s} {"구분":11s} {"얼룩%":>7s} {"색편차%":>8s} {"질감p99":>8s}')
STATS = {}
for cid, kind in TARGETS:
    fs = sorted(BY_CELL[cid]); fs = fs[::max(1, len(fs) // UNI_FRAMES)][:UNI_FRAMES]
    v = np.array([surface_stats(p) for p in fs], dtype=float)
    v = v[~np.isnan(v).any(1)]
    if not len(v): print(f'{cid:>6d} {kind:11s}  (셀 영역 검출 실패)'); continue
    STATS[cid] = (kind, v.mean(0))
    mu = v.mean(0)
    print(f'{cid:>6d} {kind:11s} {mu[0]*100:>7.2f} {mu[1]*100:>8.3f} {mu[2]:>8.2f}')

# ── 분리력 판정: 정상 최댓값을 넘는 결함 셀이 몇 개인가 ──
print(f'\n{"지표":10s} {"정상 최대":>10s} {"결함 최소":>10s} {"결함 중앙":>10s}  판정')
NAMES = ('얼룩', '색편차', '질감')
WINNERS = []
for j, nm in enumerate(NAMES):
    cl = [s[1][j] for s in STATS.values() if s[0].startswith('정상')]
    df = [s[1][j] for s in STATS.values() if s[0].startswith('결함')]
    if not cl or not df: continue
    hit = sum(1 for x in df if x > max(cl))
    ok = hit == len(df)
    if ok: WINNERS.append((nm, max(cl), min(df)))
    print(f'{nm:10s} {max(cl):>10.4f} {min(df):>10.4f} {np.median(df):>10.4f}  '
          + (f'✅ 완전 분리 ({hit}/{len(df)})' if ok else
             f'부분 {hit}/{len(df)}' if hit else '분리 없음'))

print('\n' + '─' * 66)
if WINNERS:
    nm, hi, lo = WINNERS[0]
    print(f'✅ **{nm}**로 정상/결함이 갈린다 — 임계 후보 {(hi+lo)/2:.4f} '
          f'(정상 최대 {hi:.4f} < 결함 최소 {lo:.4f})')
    print("""   쓰는 법: 이 값을 **셀 단위 1차 게이트**로 두고, 통과 못 한 셀만 §1~§4로 넘긴다.
            검출기·VLM보다 1000배 싸고 §7이 못 한 분리를 해낸다.
   ⚠️ 셀 {}개 표본이다. 확정 전에 TARGETS를 늘려 재측정할 것.
   ⚠️ 정상 셀이 '깨끗해 보이는 셀'로 선별된 것이라면 이 지표는 선별 기준을 배운 것일 수 있다 —
      §8 육안 확인 결과와 같이 읽을 것.""".format(len(STATS)))
else:
    print("""🔴 표면 통계로도 안 갈린다.
   → 정상/결함의 차이가 전역 표면이 아니라 국소 결함에만 있다는 뜻이고,
     그러면 §7의 결론(검출기 분리력 부족)이 진짜 병목이다.
   ⚠️ 단 §8에서 "정상 셀"에 결함이 보였다면 이 판정도 무효다 — 갈릴 게 없었던 것이다.""")

# 🥇 정상 셀 후보 찾기 — 매니페스트로는 못 고른다. 표면이 가장 고른 셀을 순위로
#    뽑아 §8에서 눈으로 확인한다 — 판정은 사람이 하고 코드는 후보만 좁힌다.
SCAN_N, SCAN_FRAMES, SHOW = 40, 6, 10
_cand = sorted(c for c in BY_CELL if c not in DEFECT_OK and c not in CLEAN_OK)
_cand = random.Random(0).sample(_cand, min(SCAN_N, len(_cand)))
print(f'\n\n■ 셀 {len(_cand)}개 표면 얼룩 랭킹 (정상 후보 좁히기 — 라벨 미사용)')
_rank = []
for cid in _cand:
    fs = sorted(BY_CELL[cid]); fs = fs[::max(1, len(fs) // SCAN_FRAMES)][:SCAN_FRAMES]
    v = np.array([surface_stats(p) for p in fs], dtype=float)
    v = v[~np.isnan(v).any(1)]
    if len(v): _rank.append((float(v[:, 0].mean()), float(v[:, 1].mean()), cid))
_rank.sort()
print(f'  {"셀":>6s} {"얼룩%":>8s} {"색편차%":>9s}')
for sm, cr, cid in _rank[:SHOW]: print(f'  {cid:>6d} {sm*100:>8.2f} {cr*100:>9.3f}')
if _rank:
    print(f'\n  → 위 {SHOW}개를 §8의 VIEW_IDS에 넣어 육안 확인.'
          f'\n    깨끗하면 §0의 **CLEAN_OK**, 결함이면 **DEFECT_OK**에 추가하고 §7·§9 재실행.'
          f'\n    ⚠️ 0804 실측에서는 상위 10개가 전부 결함이었다 — 정상 셀이 안 나오면'
          f'\n       셀 분류는 포기하고 **evalset 점 recall**로 간다(정상 셀이 필요 없다).')
    print(f'  VIEW_IDS = {[c for _, _, c in _rank[:SHOW]]}')
